# 🎬 EvoAgent (ReAgent-V): Multi-Agent Video Understanding on Kaggle (2x GPU T4)

Notebook này đã được **vá toàn diện mọi lỗi (FlashAttention, OOM 2x T4 GPU, lệch vocab size tensor, hardcoded paths)** để bạn có thể chạy một mạch từ đầu đến cuối mà **KHÔNG CẦN add bất kỳ Data Source nào**.

### ⚙️ Hướng dẫn thiết lập Kaggle trước khi chạy:
1. Ở thanh menu bên phải (**Notebook options**):
   - **Accelerator**: Chọn **`GPU T4 x2`**
   - **Internet**: Bật **`Internet on`** (Bắt buộc để tải repo và weights từ Hugging Face)
2. Nhấn **Run All** hoặc chạy từng cell theo thứ tự.

## 📦 Bước 1: Clone Repository & Cài đặt môi trường

In [ ]:
# 1. Clone repository EvoAgent đã được fix sạch lỗi
!rm -rf /kaggle/working/EvoAgent
!git clone https://github.com/nta2112/EvoAgent.git /kaggle/working/EvoAgent

# 2. Cài đặt các thư viện cần thiết
!pip install -q --upgrade pip
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q transformers==4.40.0 accelerate==0.29.3 bitsandbytes
!pip install -q ffmpeg-python decord opencv-python librosa soundfile easydict networkx yt-dlp faiss-gpu
!pip install -q git+https://github.com/LLaVA-VL/LLaVA-NeXT.git

# 3. Tắt debugger pdb vô tình để quên trong thư viện LLaVA
import glob
for p in glob.glob("/usr/local/lib/python*/dist-packages/llava/model/builder.py"):
    with open(p, "r", encoding="utf-8") as f:
        c = f.read()
    c = c.replace("import pdb;pdb.set_trace()", "pass").replace("import pdb; pdb.set_trace()", "pass")
    with open(p, "w", encoding="utf-8") as f:
        f.write(c)

print("✅ Cài đặt môi trường và vá lỗi thư viện hoàn tất 100%!")

## 🎥 Bước 2: Tải Video mẫu để thử nghiệm
(Bạn có thể đổi URL video bên dưới hoặc upload file video của bạn lên)

In [ ]:
import os

video_path = "/kaggle/working/sample_video.mp4"

# Tải một đoạn video mẫu chuẩn định dạng MP4
if not os.path.exists(video_path):
    print("Đang tải video mẫu...")
    !curl -L -o {video_path} "https://raw.githubusercontent.com/intel-iot-devkit/sample-videos/master/person-bicycle-car-detection.mp4"
    print(f"✅ Đã tải video mẫu tại: {video_path}")
else:
    print(f"Video đã tồn tại: {video_path}")

## 🤖 Bước 3: Cấu hình Phân bổ Bộ nhớ 2x T4 GPU & Tải mô hình

In [ ]:
import sys
import os
import torch

# Thêm đường dẫn module của repo vào sys.path
module_path = "/kaggle/working/EvoAgent/ReAgent-V"
if module_path not in sys.path:
    sys.path.append(module_path)
os.chdir(module_path)

# Cấu hình đường dẫn model tải trực tiếp từ Hugging Face
# - Dùng Whisper-base (~200MB) thay vì Whisper-large (6.2GB) để triệt tiêu lỗi tràn RAM
# - Cấu hình max_memory khống chế chuẩn xác VRAM từng GPU T4
path_dict = {
    "clip_model_path": "openai/clip-vit-large-patch14-336",
    "clip_cache_dir": "/kaggle/working/cache_models",
    "whisper_model_path": "openai/whisper-base",
    "whisper_cache_dir": "/kaggle/working/cache_models",
    "llava_model_path": "lmms-lab/LLaVA-Video-7B-Qwen2",
    "llava_cache_dir": "/kaggle/working/cache_models",
    "max_memory": {0: "10.5GiB", 1: "6.5GiB", "cpu": "24GiB"},
    "attn_implementation": "sdpa",
    "torch_dtype": torch.float16
}

print("⏳ Đang khởi tạo các mô hình (LLaVA-Video-7B, CLIP, Whisper)... Quá trình này mất khoảng 2-4 phút.")
from ReAgentV import ReAgentV

qa_system = ReAgentV.load_default(path_dict)
print("✅ Khởi tạo hệ thống Multi-Agent thành công trên 2x GPU T4 (Không bị tràn VRAM, không lỗi FlashAttention)!")

## 🚀 Bước 4: Chạy Pipeline suy luận & Đánh giá Multi-Agent (EvoAgent / ReAgent-V)

In [ ]:
import json
from ReAgentV_utils.model_inference.model_inference import llava_inference
from ReAgentV_utils.prompt_builder.prompt import tool_retrieval_prompt_template

# -----------------------------------------------------
# ĐẶT CÂU HỎI VỀ VIDEO TẠI ĐÂY:
question = "Describe the main event and action happening in this video."
# -----------------------------------------------------

print(f"\n[1/5] Lấy mẫu khung hình thông minh (ECRS Sampling) cho câu hỏi: '{question}'...")
frames, key_frames, key_indices, max_frames_num, raw_video, video_tensor = (
    qa_system.load_and_sample_video(
        question=question,
        video_path=video_path
    )
)
print(f"Đã trích xuất {len(key_frames)} key-frames quan trọng nhất.")

print("\n[2/5] Kích hoạt công cụ thị giác/âm thanh đa thể thức (OCR, ASR, Detection...)... ")
modal_info, det_top_idx, USE_OCR, USE_ASR, USE_DET = qa_system.retrieve_modal_info(
    video_path=video_path,
    question=question,
    frames=key_frames,
    raw_video=raw_video,
    clip_model=qa_system.clip_model,
    clip_processor=qa_system.clip_processor,
)
print(f"Các công cụ được kích hoạt: OCR={USE_OCR}, ASR={USE_ASR}, DET={USE_DET}")

print("\n[3/5] Xây dựng Prompt & Suy luận ban đầu (Initial Answer)...")
qs = qa_system.build_multimodal_prompt(
    question=question,
    modal_info=modal_info,
    det_top_idx=det_top_idx,
    max_frames_num=max_frames_num,
    USE_DET=USE_DET,
    USE_ASR=USE_ASR,
    USE_OCR=USE_OCR,
)
initial_answer = llava_inference(qs, video_tensor)
print("\n--- INITIAL ANSWER ---")
print(initial_answer)

print("\n[4/5] Tác nhân phản biện (Critic Agent) kiểm tra điểm mù và sinh báo cáo đánh giá...")
critique_questions = qa_system.generate_critical_questions(
    question, initial_answer, modal_info, video_tensor
)

updated_infos = {}
for cq in critique_questions:
    tool_selection_prompt = tool_retrieval_prompt_template.format(question=cq)
    response_list = llava_inference(tool_selection_prompt, video=None)
    new_modal_info, new_det_top_idx, _, _, _ = qa_system.retrieve_modal_info(
        video_path=video_path,
        question=cq,
        frames=key_frames,
        raw_video=raw_video,
        clip_model=qa_system.clip_model,
        clip_processor=qa_system.clip_processor,
    )
    updated_infos[cq] = new_modal_info

context_infos = {question: modal_info, **updated_infos}
context_str = json.dumps(context_infos[question], indent=2)

eval_report = qa_system.generate_eval_report(
    question=question,
    context_info=context_str,
    initial_answer=initial_answer,
    video=video_tensor
)
print("\n--- EVALUATION & REWARD REPORT ---")
print(eval_report)

print("\n[5/5] Phản biện đa góc nhìn (Conservative, Neutral, Aggressive) -> Final Answer...")
final_answer = qa_system.get_reflective_final_answer(
    question, initial_answer, eval_report, video_tensor
)

print("\n======================================================")
print("🎉 KẾT QUẢ CUỐI CÙNG (REFLECTIVE FINAL ANSWER):")
print("======================================================")
print(final_answer)